### Title: 02_generate_tree_taxonomy
### Purpose: Generate tree taxonomies for the mixed meals and the ingredients datasets. The taxaonomy is used to build the food trees for these datasets. Ingredient codes are needed to construct the OTU table (referred to as IFC table in DietDiveR). This version takes the FDA ingredient descriptions to work with the polyphenol data (mapped with FDA-FDD descriptions rather than the FNDDS ingredient descriptions)
### Author: Jules Larke
### Date: August 26, 2025

### Load packages

In [1]:
import pandas as pd
import string

## Mixed Meals Taxonomy

### Load data

In [2]:
# NodeLabelsMCT.txt comes from DietDiveR
food_id = pd.read_csv('../../data/02/NodeLabelsMCT.txt', sep='\t')

# food_tree.txt comes from 00_generate_datasets and contains the unique foodcodes in our dataset
food_codes = pd.read_csv('../../data/00/food_tree.txt', sep='\t')

### Get the unique foodcodes and descriptions in our dataset

In [3]:
# filter food codes for what is in the data subset. load data:
subset_codes = pd.read_csv('../../data/01/insulin_resistance/wweia_mixed_meals_long.tsv', sep='\t', usecols=['foodcode'])

# get unique codes
subset_codes = subset_codes.drop_duplicates()

# filter data
food_codes = food_codes[food_codes['FoodCode'].isin(subset_codes['foodcode'])]

food_codes.copy()

,FoodCode,Main.food.description,FoodID
1,91715300,100 GRAND Bar,91715300
3,57319000,"100% Natural Cereal, plain, Quaker",57319000
4,91726420,3 MUSKETEERS Bar,91726420
9,91302020,Agave liquid sweetener,91302020
10,53420300,"Air filled fritter or fried puff, without syru...",53420300
...,...,...,...
9605,11411100,"Yogurt, whole milk, plain",11411100
9608,41435120,Zone Perfect Classic Crunch nutrition bar,41435120
9609,58301150,Zucchini lasagna (diet frozen meal),58301150
9611,75316010,"Zucchini with tomato sauce, cooked, fat not ad...",75316010


### Format text for use with DietDiveR trees

In [4]:
# get punctuation for text cleaning
punct = string.punctuation
punct = punct.replace('_', '')

# apply function to clean text. removes punctuation and replaces whitespaces with underscores: .str.replace(' ', '_')
def clean_text(text):
    text = "".join([word for word in text if word not in punct])
    return text

food_codes['Main.food.description'] = food_codes['Main.food.description'].apply(lambda x: clean_text(x))
food_codes['Main.food.description'] = food_codes['Main.food.description'].str.replace(' ', '_')

# save ingredient tree taxonomy
food_codes.to_csv('../../data/02/wweia_foodcode_taxa_clean.txt', sep='\t', index=None)

## Ingredients Taxonomy
### This code is used to generate the Food Tree taxonomies for the FNDDS ingredient descriptions with the Insulin Resistance dataset

### Load data

In [5]:
ingred_id = pd.read_csv('../../data/02/ingredient_tree_labels.txt', sep='\t')
ingred_codes = pd.read_csv('../../data/01/insulin_resistance/wweia_ingredients_long.tsv', sep='\t', usecols=['ingred_desc','ingred_code'])

### Reformat and merge data: generate a file with the ingredients descriptions, their FoodID corresponding to taxonomy and ingredient code for matching back to the main dataset

In [6]:
# rename column for merging
ingred_codes = ingred_codes.rename(columns={'ingred_desc': 'Main.food.description'})
ingred_codes = ingred_codes.rename(columns={'ingred_code': 'Ingredient code'})

# get unique ingredient descriptions
ingred_codes = ingred_codes.drop_duplicates(subset='Main.food.description')
ingred_id = ingred_id.drop_duplicates(subset='Main.food.description')

# merge node labels and ingredient codes
node_and_code = ingred_id.merge(ingred_codes, on='Main.food.description', how='left')

# select features for tree taxonomy
node_labels = node_and_code[['Level.code', 'Main.food.description']]

# create copies for text cleaning
node_labels = node_labels.copy()
node_and_code = node_and_code.copy()

### Clean text and save

In [7]:
# get punctuation for text cleaning
punct = string.punctuation
punct = punct.replace('_', '')

# apply function to clean text. removes punctuation and replaces whitespaces with underscores: .str.replace(' ', '_')
def clean_text(text):
    text = "".join([word for word in text if word not in punct])
    return text

node_labels['Main.food.description'] = node_labels['Main.food.description'].apply(lambda x: clean_text(x))
node_labels['Main.food.description'] = node_labels['Main.food.description'].str.replace(' ', '_')

# save ingredient tree taxonomy
node_labels.to_csv('../../data/02/node_labels_clean.txt', sep='\t', index=None)

### Now we output the full taxaonomy file to build the tree. This contains the higher taxonomy levels in addition to the leaf nodes.

In [8]:
# drop NAs which correspond to levels that are not ingredients (leaves) and have no ingredient codes
node_and_code = node_and_code.dropna()
node_and_code['Ingredient code'] = node_and_code['Ingredient code'].astype(int) 

# rename
node_and_code = node_and_code.rename(columns={'Level.code':'FoodID'})

# as above, apply function to clean text
def clean_text(text):
    text = "".join([word for word in text if word not in punct])
    return text

node_and_code['Main.food.description'] = node_and_code['Main.food.description'].apply(lambda x: clean_text(x))
node_and_code['Main.food.description'] = node_and_code['Main.food.description'].str.replace(' ', '_')

# save ingredient taxonomy linked codes for OTU tables
node_and_code.to_csv('../../data/02/wweia_ingredient_taxa_clean.txt', sep='\t', index=None)

### We will also create a taxonomy that takes the FDA ingredient descriptions to work with the polyphenol data (mapped with FDA-FDD descriptions rather than the FNDDS ingredient descriptions)

### Load data

In [9]:
# node_labels_clean.txt was created by JL to generate a food tree specific to ingredients
# node_labels = pd.read_csv('../../data/02/node_labels_clean.txt', sep='\t') (already loaded from above)
# node_and_code = pd.read_csv('../../data/02/wweia_ingredient_taxa_clean.txt', sep='\t') (already loaded from above)
fda_ingredients = pd.read_csv('../../data/00/wweia_dataset/wweia_ingredients_fda_desc_recalls_2023.csv', usecols=['fda_desc', 'ingred_code'])

### Reformat and merge data: generate a file with the ingredients descriptions, their FoodID corresponding to taxonomy and ingredient code for matching back to the main dataset

In [10]:
# get unique ingredient codes
fda_ingredients = fda_ingredients.copy()

fda_ingredients = fda_ingredients.drop_duplicates(subset='ingred_code')
fda_ingredients['ingred_code'] = fda_ingredients['ingred_code'].astype(int)

# rename and merge
node_and_code = node_and_code.rename(columns={'ingred_code':'Ingredient code'})
fda_ingredients = fda_ingredients.rename(columns={'fda_desc':'ingred_desc', 'ingred_code':'Ingredient code'})
fda_node_and_code = fda_ingredients.merge(node_and_code, on='Ingredient code')

# drop and rename to match correct format
fda_node_and_code = fda_node_and_code.drop(columns='Main.food.description')
fda_node_and_code = fda_node_and_code.rename(columns={'ingred_desc':'Main.food.description'})

### Clean text and save

In [11]:
# get punctuation for text cleaning
punct = string.punctuation
punct = punct.replace('_', '')

# apply function to clean text. removes punctuation and replaces whitespaces with underscores: .str.replace(' ', '_')
def clean_text(text):
    text = "".join([word for word in text if word not in punct])
    return text

fda_node_and_code['Main.food.description'] = fda_node_and_code['Main.food.description'].apply(lambda x: clean_text(x))
fda_node_and_code['Main.food.description'] = fda_node_and_code['Main.food.description'].str.replace(' ', '_')

# sort values
fda_node_and_code['FoodID'] = fda_node_and_code['FoodID'].astype(str)
fda_node_and_code = fda_node_and_code.sort_values('FoodID')

# save ingredient tree taxonomy
fda_node_and_code[['FoodID', 'Main.food.description', 'Ingredient code']].to_csv('../../data/02/fda_wweia_ingredient_taxa_clean.txt', sep='\t', index=None)

### Now we output the full taxaonomy file to build the tree. This contains the higher taxonomy levels in addition to the leaf nodes.

In [12]:
# convert back to int for filtering
fda_node_and_code['FoodID'] = fda_node_and_code['FoodID'].astype(int)

# get the taxa that are not L1 (leaf nodes)
non_l1_nodes = node_labels[~node_labels['Level.code'].isin(fda_node_and_code['FoodID'])]

# rename for formatting
fda_node_and_code = fda_node_and_code.rename(columns={'FoodID':'Level.code'})

# select columns
fda_node_and_code = fda_node_and_code[['Level.code', 'Main.food.description']]

# concat the rows
fda_node_lables = pd.concat([fda_node_and_code, non_l1_nodes])

# sort values
fda_node_lables['Level.code'] = fda_node_lables['Level.code'].astype(str)
fda_node_lables = fda_node_lables.sort_values('Level.code')

# save
fda_node_lables.to_csv('../../data/02/fda_node_labels_clean.txt', sep='\t', index=None)